# Przydzielenie 39 uczestnikom jednej z 3 grup: DE, IT i SSH - metoda Hugging Face

## Kod dzieli transkrypcje rozmów uczestników na fragmenty tekstu (chunki), wykorzystuje model HerBERTa z biblioteki Hugging Face Transformers oraz Group K-Fold Cross-Validation do podziału danych na train i test oraz ochronę możliwego powtórzenia danych, co mogłoby skutkować przeuczeniem modelu.

### Problem: HerBERT przetwarza tylko max 512 max_length tekstu, więc ogólnym planem jest wpierw podział tekstów każdego uczestnika na chunki, które w końcowym etapie połączy się i da to pełną informację, czy dany uczestnik został poprawnie zaklasyfikowany.

### Ważne, najlepiej model trenować na GPU, gdyż wtedy trenowanie pójdzie szybciej.

In [ ]:
import io
import zipfile
import urllib.request
from pathlib import Path
import pandas as pd

# --- Params ---
participant_id = 'MW_SSH_05'
transcripts_dir = Path('transcripts')  # extracted CSVs will be placed here
transcripts_dir.mkdir(parents=True, exist_ok=True)

# --- Ensure TRANSCRIPTS.zip is downloaded & extracted ---
zip_url = 'https://zenodo.org/records/15222484/files/TRANSCRIPTS.zip?download=1'
expected_csv = transcripts_dir / f'{participant_id}.csv'

if not expected_csv.exists():
    print('Downloading TRANSCRIPTS.zip from Zenodo...')
    with urllib.request.urlopen(zip_url) as resp:
        data = resp.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        zf.extractall(transcripts_dir)
    print(f'Extracted transcripts to: {transcripts_dir.resolve()}')
else:
    print('Transcripts already present, skipping download.')

# --- Load problems and responses from Zenodo ---
problems = pd.read_csv('https://zenodo.org/records/15222484/files/PROBLEMS.csv')
responses = pd.read_csv('https://zenodo.org/records/15222484/files/PROBLEMS_RESPONSES.csv')
merged_problems = pd.merge(responses, problems, on='problem_id', how='inner')

# Use column names as defined in the dataset docs
prediction_features = ['participant_decision', 'participant_certaintity', 'model_class', 'model_probability']
print(merged_problems[merged_problems['participant_id'] == participant_id][prediction_features])

# --- Retrieve transcript text for problem __P1__ for the participant ---
transcripts = pd.read_csv(expected_csv)
# Forward-fill problem markers within the participant's transcript
if 'problem_id' in transcripts.columns:
    transcripts['problem_id'] = transcripts['problem_id'].ffill()
    subset = transcripts[transcripts['problem_id'] == '__P1__']
else:
    subset = pd.DataFrame()

# Show key columns if present
cols = [c for c in ['speaker_id', 'slide_id', 'question_id', 'problem_id', 'text'] if c in subset.columns]
print(subset[cols] if cols else subset)

Transcripts already present, skipping download.
   participant_decision participant_certaintity model_class model_probability
57              trujący           średnio pewny     jadalny              0,54
58              jadalny           średnio pewny     jadalny              0,95
59              jadalny      zdecydowanie pewny     trujący                 1
    speaker_id slide_id question_id problem_id  \
180  MW_SSH_05      NaN         NaN     __P1__   
181         MW      NaN         NaN     __P1__   
182  MW_SSH_05      NaN         NaN     __P1__   

                                                  text  
180        Ok. Średnica 17,30... Mogę po całej kartce?  
181               Jasne, proszę sobie mazać spokojnie.  
182  Ok to wróćmy do… To była wysokość, średnica ka...  


# Embedding

In [ ]:
import io
import zipfile
import re
import urllib.request
from pathlib import Path
import pandas as pd
import numpy as np

import torch
from torch.nn import CrossEntropyLoss

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer

## Przygotowywanie danych

In [ ]:
# CHUNKS

def chunk_text(text, chunk_size=128, overlap=32):

    words = text.split()
    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size
        chunk = words[start:end]

        if len(chunk) > 20:
            chunks.append(" ".join(chunk))

        start += chunk_size - overlap

    return chunks

# BUILD DATASET

rows = []

for csv_file in transcripts_dir.glob("*.csv"):

    df = pd.read_csv(csv_file)
    df = df[df["speaker_id"].astype(str) != "PK"]

    df["speaker_id"] = df["speaker_id"].astype(str)
    df["participant_num"] = df["speaker_id"].str.extract(r"(\d+)$")

    filename = csv_file.stem

    # SPLIT FILE PK_DE_11-12
    if filename == "PK_DE_11-12":

        parts = [
            ("PK_DE_11", df[df["speaker_id"].str.contains("11", na=False)]),
            ("PK_DE_12", df[df["speaker_id"].str.contains("12", na=False)])
        ]

    else:
        parts = [(filename, df)]

    for participant_id, part_df in parts:

        label = participant_id.split("_")[1]
        participant_key = participant_id.split("_")[2]

        # ONLY PARTICIPANT SPEECH (DELETE MODERATORS)
        participant_rows = part_df[
            part_df["participant_num"] == participant_key
        ]

        texts = participant_rows["text"].dropna().astype(str).tolist()

        # CLEANING
        cleaned_texts = []
        for t in texts:

            t = t.lower()
            t = re.sub(r"http\S+", "", t)
            t = re.sub(r"\s+", " ", t).strip()

            if len(t.split()) > 2:
                cleaned_texts.append(t)

        texts = cleaned_texts

        # SPEACH CHUNKS
        full_text = " ".join(texts)

        if len(full_text.strip()) == 0:
            continue

        chunks = chunk_text(full_text)

        for chunk in chunks:

            rows.append({
                "participant_id": participant_id,
                "text": chunk,
                "label": label
            })

dataset_df = pd.DataFrame(rows)

print(dataset_df.head())
print("\nDataset size:")
print(dataset_df.shape)
print("\nClass distribution:")
print(dataset_df["label"].value_counts())

# LABELS TO NUMBERS

label2id = {"DE": 0, "SSH": 1, "IT": 2}
id2label = {v: k for k, v in label2id.items()}

dataset_df["label_id"] = dataset_df["label"].map(label2id)

  participant_id                                               text label
0       PK_DE_07  tak zapoznałem się, aprobuję. mhm. może trochę...    DE
1       PK_DE_07  prostu tak, że są grzyby, które de facto są ja...    DE
2       PK_DE_07  tak, z tego, co tam było napisane? no bo na pr...    DE
3       PK_DE_07  widzę, że ich to jest najwięcej, tak… eee… kol...    DE
4       PK_DE_07  no to przyrasta na całej grubości, ale zbiega ...    DE

Dataset size:
(2014, 3)

Class distribution:
label
SSH    1011
IT      513
DE      490
Name: count, dtype: int64


## Wykorzystywane elementy

In [ ]:
# GROUP K-FOLD

X = dataset_df["text"].values
y = dataset_df["label_id"].values
groups = dataset_df["participant_id"].values

### TU ZMIENIASZ ILOSC FOLDOW ###
gkf = GroupKFold(n_splits=3)

# TOKENIZER (MODEL)

MODEL_NAME = "allegro/herbert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# HUGGING_FACE

class ConversationDataset(torch.utils.data.Dataset):

    def __init__(self, texts, labels):

        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            ### TU ZMIENIASZ ILOSC TEKSTU NA CHUNK ###
            max_length=256
        )

        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

# METRICS

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

## Trenowanie

In [ ]:
# TRAINER

class WeightedTrainer(Trainer):

    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):

        labels = inputs["labels"]
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# G-FOLD TRAIN LOOP

all_results = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):

    print(f"\nFOLD {fold}")

    train_df = dataset_df.iloc[train_idx]
    val_df = dataset_df.iloc[val_idx]

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(train_df["label_id"]),
        y=train_df["label_id"]
    )

    class_weights = torch.tensor(class_weights, dtype=torch.float)

    train_dataset = ConversationDataset(
        train_df["text"].tolist(),
        train_df["label_id"].tolist()
    )

    val_dataset = ConversationDataset(
        val_df["text"].tolist(),
        val_df["label_id"].tolist()
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
        id2label=id2label,
        label2id=label2id
    )

    training_args = TrainingArguments(
        output_dir=f"./results_fold_{fold}",

        learning_rate=2e-5,

        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,

        ### TU ZMIENIASZ ILOSC EPOCH ###
        num_train_epochs=5,

        weight_decay=0.01,

        logging_steps=10,

        eval_strategy="epoch",
        save_strategy="epoch",

        load_best_model_at_end=True,

        metric_for_best_model="f1_macro",
        greater_is_better=True,

        save_total_limit=1
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        class_weights=class_weights,
        compute_metrics=compute_metrics
    )

    trainer.train()

    preds = trainer.predict(val_dataset)

    logits = preds.predictions
    true = preds.label_ids
    pred = np.argmax(logits, axis=1)

    fold_df = pd.DataFrame({
        "participant_id": val_df["participant_id"].values,
        "true_label": true,
        "predicted_label": pred,
        "correct": true == pred
    })

    all_results.append(fold_df)

    print(classification_report(true, pred, labels=[0, 1, 2], target_names=["DE", "SSH", "IT"], zero_division=0))

    cm = confusion_matrix(true, pred)

    print("\nConfusion Matrix:")
    print(cm)


FOLD 0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly in

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.586997,0.852203,0.463378,0.567784
2,0.619370,2.744564,0.463378,0.479572
3,0.537208,3.194299,0.439462,0.462042
4,0.007039,4.237276,0.463378,0.486289
5,0.000917,4.304931,0.467862,0.504082


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

              precision    recall  f1-score   support

          DE       0.94      0.91      0.93       187
         SSH       0.81      0.78      0.80       385
          IT       0.36      0.43      0.39        98

    accuracy                           0.77       670
   macro avg       0.70      0.71      0.70       670
weighted avg       0.78      0.77      0.77       670


Confusion Matrix:
[[170  17   0]
 [  8 301  76]
 [  2  54  42]]


## Wyniki

In [ ]:
final_df = pd.concat(all_results, ignore_index=True)

final_df["true_label_name"] = final_df["true_label"].map(id2label)
final_df["predicted_label_name"] = final_df["predicted_label"].map(id2label)

participant_predictions = []

for participant_id in final_df["participant_id"].unique():

    participant_rows = final_df[
        final_df["participant_id"] == participant_id
    ]

    true_label = participant_rows["true_label"].iloc[0]

    predicted_label = (
        participant_rows["predicted_label"]
        .mode()[0]
    )

    participant_predictions.append({
        "participant_id": participant_id,
        "true_label": true_label,
        "predicted_label": predicted_label,
        "correct": true_label == predicted_label
    })

participant_df = pd.DataFrame(participant_predictions)

# % OF ACCURACY

chunk_accuracy = accuracy_score(
    participant_df["true_label"],
    participant_df["predicted_label"]
)

print("\n==============================")
print(f"CHUNK ACCURACY: {chunk_accuracy * 100:.2f}%")
print("==============================")

print("\nACCURACY PER CLASS FOR CHUNKS:")

for class_id, class_name in id2label.items():

    class_subset = final_df[
        final_df["true_label"] == class_id
    ]

    class_accuracy = accuracy_score(
        class_subset["true_label"],
        class_subset["predicted_label"]
    )

    print(
        f"{class_name}: "
        f"{class_accuracy * 100:.2f}% "
        f"({len(class_subset)} chunks)"
    )

participant_accuracy = accuracy_score(
    participant_df["true_label"],
    participant_df["predicted_label"]
)

correct_participants = participant_df["correct"].sum()
total_participants = len(participant_df)

print("\n==============================")
print(
    f"PARTICIPANTS ACCURACY: "
    f"{participant_accuracy * 100:.2f}% "
    f"({correct_participants}/{total_participants})"
)
print("==============================")

print("\nACCURACY PER CLASS FOR PARTICIPANTS:")

for class_id, class_name in id2label.items():

    class_subset = participant_df[
        participant_df["true_label"] == class_id
    ]

    class_accuracy = accuracy_score(
        class_subset["true_label"],
        class_subset["predicted_label"]
    )

    correct_count = class_subset["correct"].sum()

    total_count = len(class_subset)

    print(
        f"{class_name}: "
        f"{class_accuracy * 100:.2f}% "
        f"({correct_count}/{total_count} participants correct)"
    )

print("\nCLASSIFICATION REPORT:\n")

print(
    classification_report(
        participant_df["true_label"],
        participant_df["predicted_label"],
        target_names=["DE", "SSH", "IT"]
    )
)

print("\nCONFUSION MATRIX:\n")

cm = confusion_matrix(
    participant_df["true_label"],
    participant_df["predicted_label"]
)

print(cm)

csv_df = participant_df.copy()

csv_df["true_label_name"] = csv_df["true_label"].map(id2label)
csv_df["predicted_label_name"] = csv_df["predicted_label"].map(id2label)

csv_df = csv_df[[
    "participant_id",
    "true_label_name",
    "predicted_label_name",
    "correct"
]]

csv_df.columns = [
    "participant",
    "true_label",
    "predicted_label",
    "correct"
]

csv_df.to_csv(
    "bert_gfold_results.csv",
    index=False
)

print("\nSaved: bert_gfold_results.csv")

print("\nRESULTS PREVIEW:")
print(csv_df.head())


CHUNK ACCURACY: 79.49%

ACCURACY PER CLASS FOR CHUNKS:
DE: 94.29% (490 chunks)
SSH: 77.25% (1011 chunks)
IT: 27.68% (513 chunks)

PARTICIPANTS ACCURACY: 79.49% (31/39)

ACCURACY PER CLASS FOR PARTICIPANTS:
DE: 100.00% (13/13 participants correct)
SSH: 94.44% (17/18 participants correct)
IT: 12.50% (1/8 participants correct)

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

          DE       1.00      1.00      1.00        13
         SSH       0.71      0.94      0.81        18
          IT       0.50      0.12      0.20         8

    accuracy                           0.79        39
   macro avg       0.74      0.69      0.67        39
weighted avg       0.76      0.79      0.75        39


CONFUSION MATRIX:

[[13  0  0]
 [ 0 17  1]
 [ 0  7  1]]

Saved: bert_gfold_results.csv

RESULTS PREVIEW:
  participant true_label predicted_label  correct
0    MZ_IT_06         IT             SSH    False
1    MK_IT_06         IT             SSH    False
2   DR_SSH_

## Model dobrze radzi sobie z grupą DE, natomiast ma problemy z IT, przyporządkowując jej najczęściej SSH. To oznacza, że transkrypcja IT i SSH jest bardzo zbliżona do siebie + IT jest najmniej liczącą grupą porównując najbardziej liczną grupę SSH, co może dać problemy z poprawną klasyfikacją tej grupy.